## Theory

Naïve Bayes is a supervised machine learning classification algorithm based on Bayes' Theorem. It assumes that all input features are conditionally independent of one another given the target class. It calculates the probability of each class based on the given input features and predicts the class with the highest probability.

Bayes' Theorem is expressed as:

$$
P(C \mid X) = \frac{P(X \mid C)\,P(C)}{P(X)}
$$

where:

- $P(C \mid X)$ is the posterior probability of class $C$ given input $X$.
- $P(X \mid C)$ is the likelihood of observing input $X$ given class $C$.
- $P(C)$ is the prior probability of class $C$.
- $P(X)$ is the evidence or probability of the input.

In this practical, the Naïve Bayes classifier uses **Outlook** (Sunny, Overcast, or Rainy) and **Windy** (Yes or No) as input features to predict **Play** (Yes or No). The model is trained using the given 10-record dataset. It predicts the class for the instance Outlook = Rainy and Windy = Yes, and also accepts new user inputs to display the predicted class and class probabilities.


In [1]:
import pandas as pd
import math

data = {
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rainy', 'Rainy',
        'Rainy', 'Overcast', 'Sunny', 'Rainy', 'Overcast'
    ],
    'Windy': [
        'No', 'Yes', 'No', 'No', 'No',
        'Yes', 'Yes', 'No', 'Yes', 'No'
    ],
    'Play': [
        'No', 'No', 'Yes', 'Yes', 'Yes',
        'No', 'Yes', 'Yes', 'Yes', 'Yes'
    ]
}

df = pd.DataFrame(data)

print("Dataset created successfully!")
print("Dataset shape:", df.shape)

df

Dataset created successfully!
Dataset shape: (10, 3)


,Outlook,Windy,Play
0,Sunny,No,No
1,Sunny,Yes,No
2,Overcast,No,Yes
3,Rainy,No,Yes
4,Rainy,No,Yes
5,Rainy,Yes,No
6,Overcast,Yes,Yes
7,Sunny,No,Yes
8,Rainy,Yes,Yes
9,Overcast,No,Yes


In [2]:
total_records = len(df)

class_counts = df['Play'].value_counts().to_dict()

prior = {}

for cls in ['Yes', 'No']:
    prior[cls] = class_counts.get(cls, 0) / total_records

print("Total records:", total_records)
print("\nPrior Probabilities:")

for cls in ['Yes', 'No']:
    print(f"P(Play={cls}) = {prior[cls]:.4f}")

Total records: 10

Prior Probabilities:
P(Play=Yes) = 0.7000
P(Play=No) = 0.3000


In [3]:
features = ['Outlook', 'Windy']
classes = ['Yes', 'No']

# Possible values of each feature
feature_values = {
    feature: sorted(df[feature].unique().tolist())
    for feature in features
}

# Count records belonging to each class
class_counts = df['Play'].value_counts().to_dict()

# Calculate conditional probabilities using Laplace smoothing
conditional = {}

for feature in features:
    conditional[feature] = {}

    for value in feature_values[feature]:
        conditional[feature][value] = {}

        for cls in classes:
            matching_count = len(
                df[(df[feature] == value) & (df['Play'] == cls)]
            )

            denominator = class_counts.get(cls, 0) + len(feature_values[feature])

            conditional[feature][value][cls] = (
                matching_count + 1
            ) / denominator

print("Conditional Probabilities:\n")

for feature in features:
    print(feature)

    for value in feature_values[feature]:
        for cls in classes:
            probability = conditional[feature][value][cls]
            print(f"P({feature}={value} | Play={cls}) = {probability:.4f}")

    print()

Conditional Probabilities:

Outlook
P(Outlook=Overcast | Play=Yes) = 0.4000
P(Outlook=Overcast | Play=No) = 0.1667
P(Outlook=Rainy | Play=Yes) = 0.4000
P(Outlook=Rainy | Play=No) = 0.3333
P(Outlook=Sunny | Play=Yes) = 0.2000
P(Outlook=Sunny | Play=No) = 0.5000

Windy
P(Windy=No | Play=Yes) = 0.6667
P(Windy=No | Play=No) = 0.4000
P(Windy=Yes | Play=Yes) = 0.3333
P(Windy=Yes | Play=No) = 0.6000



In [4]:
def predict_naive_bayes(outlook, windy):
    input_values = {
        'Outlook': outlook,
        'Windy': windy
    }

    scores = {}

    for cls in classes:
        score = prior[cls]

        for feature, value in input_values.items():
            score *= conditional[feature][value][cls]

        scores[cls] = score

    total_score = sum(scores.values())

    probabilities = {
        cls: scores[cls] / total_score
        for cls in classes
    }

    predicted_class = max(probabilities, key=probabilities.get)

    return predicted_class, probabilities

In [5]:
outlook = 'Rainy'
windy = 'Yes'

prediction, probabilities = predict_naive_bayes(outlook, windy)

print("Given Instance:")
print("Outlook:", outlook)
print("Windy:", windy)

print("\nPredicted Play:", prediction)

print("\nClass Probabilities:")
for cls in classes:
    print(f"P(Play={cls} | X) = {probabilities[cls]:.4f} "
          f"({probabilities[cls] * 100:.2f}%)")

Given Instance:
Outlook: Rainy
Windy: Yes

Predicted Play: Yes

Class Probabilities:
P(Play=Yes | X) = 0.6087 (60.87%)
P(Play=No | X) = 0.3913 (39.13%)


In [6]:
print("Enter values for a new prediction.")

while True:
    outlook = input("Enter Outlook (Sunny/Overcast/Rainy): ").strip().title()

    if outlook in feature_values['Outlook']:
        break

    print("Invalid Outlook! Please enter Sunny, Overcast, or Rainy.")

while True:
    windy = input("Is it Windy? (Yes/No): ").strip().title()

    if windy in feature_values['Windy']:
        break

    print("Invalid Windy value! Please enter Yes or No.")

prediction, probabilities = predict_naive_bayes(outlook, windy)

print("\n--- Prediction Result ---")
print("Outlook:", outlook)
print("Windy:", windy)
print("Predicted Play:", prediction)

print("\nClass Probabilities:")
for cls in classes:
    print(f"Play = {cls}: {probabilities[cls]:.4f} "
          f"({probabilities[cls] * 100:.2f}%)")

Enter values for a new prediction.

--- Prediction Result ---
Outlook: Rainy
Windy: Yes
Predicted Play: Yes

Class Probabilities:
Play = Yes: 0.6087 (60.87%)
Play = No: 0.3913 (39.13%)
